In [ ]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests

In [ ]:
file_list = [
  './app/data/raw/rapido.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': True, 'report': False},
  'ROUTER': {'test': True, 'report': False},
  'GROUNDING': {'test': True, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report_RAPIDO',
  
  'REFORMULATE': {'test': True, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [ ]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path).replace('\\', '/')

In [ ]:
import json

response_file_path = './app/data/processed/outcomes/outcome_20260412-154829.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260406-153646.json'

In [ ]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
  )

In [ ]:
from collections import Counter
from itertools import chain

def foundrys_tests(data: list[dict]) -> dict:
  links = []
  result = {}
  for item in data:
    
    if item is None or 'ok' in item:
      continue
    
    if 'router' in item['partial_answers']:
      node_metadata = item['node_metadata']
      route = item['partial_answers'].get('router', {'route': ''}).get('route', '')
      
      ref_link = node_metadata.get('reformulate', '').get('endpoint'.split(':')[0], '')
      tri_link = node_metadata.get('triage').get('endpoint'.split(':')[0], '')
      
      rou_link = node_metadata.get('router').get('endpoint'.split(':')[0], '')
      per_link = node_metadata.get('personality', {'endpoint': 'x'}).get('endpoint', 'x')
      gro_link = node_metadata.get('grounding', {'endpoint': 'x'}).get('endpoint', 'x')
      
      ree_link = node_metadata.get(f'agent_{route}', {'retrieve_embeddings': {'endpoint': 'x'}}).get('retrieve_embeddings',{'endpoint': 'x'}).get('endpoint', 'x')
      rag_link = node_metadata.get(f'agent_{route}', {'rag_answer': {'endpoint': 'x'}}).get('rag_answer',{'endpoint': 'x'}).get('endpoint', 'x')

      
      links.append([
        ref_link.split(':')[0], 
        tri_link, 
        rou_link, 
        per_link.split(':')[0], 
        gro_link.split(':')[0], 
        ree_link.split(':')[0], 
        rag_link.split(':')[0]
        ])

  total_counts = Counter(chain.from_iterable(links))
  del total_counts['x']
  
  total = total_counts.total()
  result['total'] = total
  
  for i in range(len(total_counts)):    
    result[f'{i}'] = {
      'count': total_counts[f'{i}'],
      'percentage': round(total_counts[f'{i}'] * 100 / total, 2)
    }
  
  return result

In [55]:
foudrys_result = foundrys_tests(data=responses)
results_foundry = foudrys_result
print(results_foundry)

{'total': 56, '0': {'count': 24, 'percentage': 42.86}, '1': {'count': 2, 'percentage': 3.57}, '2': {'count': 1, 'percentage': 1.79}, '3': {'count': 3, 'percentage': 5.36}, '4': {'count': 2, 'percentage': 3.57}, '5': {'count': 0, 'percentage': 0.0}, '6': {'count': 2, 'percentage': 3.57}, '7': {'count': 3, 'percentage': 5.36}, '8': {'count': 1, 'percentage': 1.79}, '9': {'count': 2, 'percentage': 3.57}, '10': {'count': 0, 'percentage': 0.0}}


In [39]:
for i in responses:
  node_metadata = i.get('node_metadata')
  print(node_metadata.keys())
  print(node_metadata.get('router').get('endpoint'))

dict_keys(['blocked_list', 'reformulate', 'triage', 'router', 'blocked_list_agent', 'agent_call', 'personality', 'grounding', 'join_triage_router', 'agent_tramites', 'join_personality_grounding'])
None
dict_keys(['blocked_list', 'reformulate', 'triage', 'router', 'blocked_list_agent', 'agent_call', 'personality', 'grounding', 'join_triage_router', 'agent_tramites', 'join_personality_grounding'])
None
dict_keys(['blocked_list', 'reformulate', 'triage', 'router', 'blocked_list_agent', 'agent_call', 'personality', 'grounding', 'join_triage_router', 'agent_accesibilidad', 'join_personality_grounding'])
None
dict_keys(['blocked_list', 'reformulate', 'triage', 'router', 'blocked_list_agent', 'agent_call', 'personality', 'grounding', 'join_triage_router', 'agent_accesibilidad', 'join_personality_grounding'])
None
dict_keys(['blocked_list', 'reformulate', 'triage', 'router', 'blocked_list_agent', 'agent_call', 'personality', 'grounding', 'join_triage_router', 'agent_descubrir', 'join_personali